# Mecanismo biela-manivela

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fralfaro/dataviz/blob/main/animaciones/SLIDER_CRANK_MECHANISM/slider_crank_mec.ipynb)

Representa la geometría y el movimiento de un mecanismo biela-manivela mediante objetos y restricciones geométricas.

**Qué aprenderás:** usar primitivas de Manim, controlar escenas con clases y leer el código de animación paso a paso.

## Cómo usar este notebook

1. Abre el notebook en Google Colab con el botón superior.
2. Ejecuta primero la celda de instalación y reinicia el entorno si Colab lo solicita.
3. Ejecuta después las escenas de Manim de forma individual para renderizar cada animación.
4. Cambia parámetros, colores o fórmulas y vuelve a ejecutar la escena para experimentar.

Manim renderiza los videos dentro de la salida de la celda mediante su magia de Jupyter. Los archivos del repositorio se mantienen como ejemplos didácticos; revisa cada escena antes de reutilizarla en un proyecto propio.

Repositorio: [https://github.com/fralfaro/dataviz](https://github.com/fralfaro/dataviz)

## 1. Instalación y preparación

Ejecuta esta celda una sola vez por sesión de Colab. Incluye Manim, soporte de Cairo/LaTeX y las dependencias usadas por los ejemplos.

In [ ]:
# Dependencias para ejecutar Manim en Google Colab.
# La celda sigue la guía oficial de instalación de Manim para Colab.
!sudo apt-get update -qq
!sudo apt-get install -y -qq libcairo2-dev texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science tipa libpango1.0-dev
!pip install -q manim mpmath IPython==8.21.0

# Si Colab solicita reiniciar el entorno de ejecución, hazlo y vuelve a ejecutar desde aquí.


## 3. Escenas y conceptos

Las siguientes celdas contienen la configuración, las definiciones de escena y los renders interactivos.

In [ ]:
# Paso 1: código de la escena o experimento.

from manim import *
config.media_embed = True
config.media_width = "100%"
_RI = "-v WARNING -qm --progress_bar None --disable_caching Example"

## Theory
* **Book: Design of Machinery - Robert L. Norton - 6th Edition**
* **Chapter: 4.10, p. 196**

![image](https://raw.githubusercontent.com/fralfaro/dataviz/main/animaciones/SLIDER_CRANK_MECHANISM/image.png)

$$\theta_{31} = \arcsin\left(\frac{a \sin \theta_2 - c}{b}\right)$$

$$d = a \cos \theta_2 - b \cos \theta_3$$

$$\theta_{32} = \arcsin\left(-\frac{a \sin \theta_2 - c}{b}\right) + \pi$$

In the particular case where the piston is always on the right side of the system we can simplify $\theta_3$ in this way:

$$\theta_3 = \arcsin\left(\text{sgn}(\theta_2) \cdot \frac{a \sin \theta_2 - c}{b}\right)$$

$\color{cyan}\rm Remark$: Remember that $b$ is negative in this case.



## Step 1: Values to be defined

* $a$: Rod's length
* $b$: Crank's length
* $c$: Offset between origin and piston $y$ position
* $\theta_2$: Rod's angle
* $\theta_3$: Crank's angle
* $A$: Rod end point
* $B$: Piston position
* $O_2$: Rod start point (origin)

$\theta_1=0$ and $\theta_4=90^\circ$

## Step 2: Constants and values that change over time

**Constants**: $a, b, c, O_2$.

**Variables**: $\theta_2, \theta_3$, $A$, $B$.

## Step 3: Define functions

Rod end point:
$\displaystyle A(O_2, \theta_2, a) = O_2 + a * (\cos(\theta_2), \sin(\theta_2))$

Piston position - simplified version:
$\displaystyle B(O_2, d, c) = O_2 + (d, c)$

Piston position - complete version:
$\displaystyle B(O_2, \theta_2, \theta_3, a, b, c) = O_2 + (a * \cos\theta_2 - b * \cos\theta_3, c)$

Crank angle:
$\displaystyle \theta_3(\theta_2, a, b, c) = \arcsin\left(\text{sgn}(\theta_2) \cdot \frac{a \sin \theta_2 - c}{b}\right)$

Then, the *main* variable is $\color{cyan}\theta_2$.

## Step 4: Define Mobjects

* Rod
* Crank
* Piston
* Piston's floor
* Joints (origin-rod, rod-crank, crank-piston)

## Step 5: Start code with methods


In [ ]:
# Paso 5: código de la escena o experimento.

class Example(Scene):
  def get_rod_end(self, origin, t_2, length): # length = a, t_2 = theta_2
    return origin + length * np.array((np.cos(t_2), np.sin(t_2), 0))
  
  def get_piston_coord(self, origin, t_2, t_3, a, b, c):
    d = a * np.cos(t_2) - b * np.cos(t_3)
    return origin + RIGHT * d + UP * c
  
  def get_theta_3(self, t_2, a, b, c):
    return np.arcsin(
      np.sign(t_2) * (a * np.sin(t_2) - c) / b
    )

  def construct(self):
    # 1. Define constants
    a = 2     # rod length
    b = -6.5  # crank length, negative by vector definition
    c = 1.5   # offset length
    theta_start = 70 * DEGREES # theta_2 start
    theta_end   = theta_start - 2.5 * TAU # theta_2 end
    origin = np.array((-4, -0.5, 0)) # O_2
    colors = {"rod": RED, "piston": GREEN, "crank": PURPLE}

    # 2. Define static or non-transformable objects
    floor_line = Line(LEFT, RIGHT).set_length(8)
    joint_origin       = Dot(origin).scale(0.8) # O_2
    joint_rod_crank    = joint_origin.copy()    # A
    joint_crank_piston = joint_origin.copy()    # B

    # 3. Define changing objects
    rod    = Line(color=colors["rod"], stroke_width=10)
    crank  = Line(color=colors["crank"], stroke_width=10)
    piston = Rectangle(colors["piston"], 1, 1.5, fill_opacity=0.5)

    # 4. Define groups
    grp = VGroup(rod, crank, piston, joint_rod_crank, joint_crank_piston)

    # 5. Define updaters
    def updater(grp:VGroup, alpha:float):
      rod, crank, piston, jrc, jcp = grp
      t_2 = interpolate(theta_start, theta_end, alpha)
      rod.put_start_and_end_on(
        origin, self.get_rod_end(origin, t_2, a)
      )
      t_3 = self.get_theta_3(t_2, a, b, c)
      piston.move_to(
        self.get_piston_coord(origin, t_2, t_3, a, b, c)
      )
      crank.put_start_and_end_on(
        rod.get_end(), piston.get_center()
      )
      jrc.move_to(rod.get_end())
      jcp.move_to(piston.get_center())
    
    # 6. Call the updaters if necessary
    updater(grp, 0)

    # 7. Position static objects or objects that depend on others
    floor_line.next_to(piston, DOWN, buff=0).shift(LEFT)

    # 8. Add the intial objects
    self.add(grp, joint_origin.set_z_index(10))
    self.bring_to_back(floor_line)

    # 9. Start anims
    self.wait(0.5)
    self.play(
      UpdateFromAlphaFunc(grp, updater),
      run_time=8,
      rate_func=double_smooth
    )
    self.wait()

%manim $_RI